# Word2Vec Word Vector Model Research and Practice: Pre-trained Models and Custom Training

**Objective**: Master the principles and applications of word embeddings (Word Embeddings). By loading pre-trained models and training custom models, evaluate their performance on semantic similarity and analogy reasoning tasks, and analyze potential biases present in the models.

---
## Stage 1: Researching Pre-trained Models

### 1.1 Loading the Model and Basic Information
We use the classic pre-trained English model `word2vec-google-news-300` from the `gensim-data` library.

In [1]:
import gensim.downloader as api
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import nltk
from nltk.corpus import gutenberg
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec

In [2]:
model = api.load('word2vec-google-news-300')

print(f"Number of words: {len(model.key_to_index):,}")
print(f"Vector dimension: {model.vector_size}")

Number of words: 3,000,000
Vector dimension: 300


### 1.2 Finding Nearest Neighbor Words

In [3]:
target_words = ["car", "beautiful", "play", "computer"]
for word in target_words:
    print(f"\nTop 10 nearest neighbors of [{word}]:")
    for neighbor, sim in model.most_similar(word, topn=10):
        print(f"  - {neighbor}: {sim:.4f}")


Top 10 nearest neighbors of [car]:
  - vehicle: 0.7821
  - cars: 0.7424
  - SUV: 0.7161
  - minivan: 0.6907
  - truck: 0.6736
  - Car: 0.6678
  - Ford_Focus: 0.6673
  - Honda_Civic: 0.6627
  - Jeep: 0.6511
  - pickup_truck: 0.6441

Top 10 nearest neighbors of [beautiful]:
  - gorgeous: 0.8353
  - lovely: 0.8107
  - stunningly_beautiful: 0.7329
  - breathtakingly_beautiful: 0.7231
  - wonderful: 0.6854
  - fabulous: 0.6700
  - loveliest: 0.6613
  - prettiest: 0.6595
  - beatiful: 0.6593
  - magnificent: 0.6591

Top 10 nearest neighbors of [play]:
  - playing: 0.7668
  - plays: 0.7108
  - played: 0.6962
  - game: 0.6501
  - toplay: 0.5970
  - Playing: 0.5813
  - games: 0.5265
  - paly: 0.5261
  - score: 0.5229
  - Play: 0.5014

Top 10 nearest neighbors of [computer]:
  - computers: 0.7979
  - laptop: 0.6640
  - laptop_computer: 0.6549
  - Computer: 0.6473
  - com_puter: 0.6082
  - technician_Leonard_Luchko: 0.5663
  - mainframes_minicomputers: 0.5618
  - laptop_computers: 0.5585
  - PC:

### 1.3 Semantic Similarity Evaluation

In [4]:
import scipy.stats as stats

similarity_pairs = [
    ("car", "car", 9),
    ("computer", "laptop", 8),
    ("sun", "moon", 6),
    ("cat", "dog", 7),
    ("joy", "happiness", 8),
    ("house", "building", 7),
    ("fast", "fast", 8),
    ("red", "scarlet", 9),
    ("good", "bad", 4)

]

human_scores = []
model_scores = []

print(f"{'Word 1':<12} | {'Word 2':<12} | {'Human Score':<6} | {'Model Score':<6}")
print("-" * 48)

for w1, w2, human_score in similarity_pairs:
    if w1 in model and w2 in model:
        model_score = model.similarity(w1, w2)
        human_scores.append(human_score)
        model_scores.append(model_score)
        print(f"{w1:<12} | {w2:<12} | {human_score:<8.1f} | {model_score:<6.4f}")
    else:
        print(f"Skipping {w1}-{w2}: Some words are not in the dictionary")

# 计算 Spearman 相关系数
correlation, p_value = stats.spearmanr(human_scores, model_scores)
print(f"\nSpearman : {correlation:.4f} (p-value: {p_value:.4f})")

Word 1       | Word 2       | Human Score | Model Score
------------------------------------------------
car          | car          | 9.0      | 1.0000
computer     | laptop       | 8.0      | 0.6640
sun          | moon         | 6.0      | 0.4263
cat          | dog          | 7.0      | 0.7609
joy          | happiness    | 8.0      | 0.6183
house        | building     | 7.0      | 0.4379
fast         | fast         | 8.0      | 1.0000
red          | scarlet      | 9.0      | 0.5054
good         | bad          | 4.0      | 0.7190

Spearman : 0.3163 (p-value: 0.4069)


**Conclusion (1.3)**: The empirically measured Spearman correlation coefficient is approximately 0.3163. There are discrepancies between the similarities given by the model and human ratings, mainly because Word2Vec relies heavily on 'context overlap.' For example, `good` and `bad` as antonyms have almost identical syntactic contexts, leading the model to consider them highly similar (similarity as high as 0.7190), which exposes the model's lack of true understanding of logical opposition.  

---
### 1.4 Solving Analogy Problems
Test the specific operational logic of `king - king + queen = ?`.

In [5]:
result = model.most_similar(positive=['king', 'queen'], negative=['king'], topn=1)
print(result)

[('queens', 0.7399441599845886)]


In [6]:
def test_identity_analogy(word_a, word_b):
    try:
        result = model.most_similar(positive=[word_a, word_b], negative=[word_a], topn=1)
        print(f"{word_a:} - {word_a:} + {word_b:} => Top 3: {result[0][0]:} (sim: {result[0][1]:.4f})")
    except KeyError as e:
        print(f"Vocabulary loss: {e}")

print("--- 3 reasonable analogy answers ---")
test_identity_analogy('king', 'queen') 
test_identity_analogy('car', 'vehicle')
test_identity_analogy('man', 'doctor')

print("\n--- 2 mistakes versus the answer ---")
test_identity_analogy('eat', 'apple')
test_identity_analogy('increase', 'decrease')

--- 3 reasonable analogy answers ---
king - king + queen => Top 3: queens (sim: 0.7399)
car - car + vehicle => Top 3: SUV (sim: 0.7578)
man - man + doctor => Top 3: physician (sim: 0.7806)

--- 2 mistakes versus the answer ---
eat - eat + apple => Top 3: apples (sim: 0.7204)
increase - increase + decrease => Top 3: decreases (sim: 0.8094)


**Conclusion (1.4)**:  
When performing `A - A + B`, the vector $A$ cancels out with $-A$, and the formula degenerates to 'finding the nearest neighbor of $B$'.  
1. **Getting stuck on 'plural' (`eat - eat + apple` -> `apples`)**: The model lacks true semantic reasoning and merely mechanically finds the closest noun form (plural).  
2. **Confusion with antonyms (`increase - increase + decrease` -> `decreases`)**: This again proves that the model cannot distinguish highly co-occurring antonyms.

## Stage 2: Training a Custom Word2Vec Model
Train using the NLTK Gutenberg classical literature corpus.

In [7]:
nltk.download('gutenberg')
nltk.download('punkt')

print("Loading and preprocessing the corpus...")
# Get all sentences from the Gutenberg corpus
raw_sentences = gutenberg.sents()

# Preprocessing: convert to lowercase, remove punctuation, tokenize (keep words with length >= 2)
# simple_preprocess by default converts words to lowercase and removes non-alphabetic characters
sentences = [simple_preprocess(" ".join(sent)) for sent in raw_sentences]

print(f"Preprocessing complete! A total of {len(sentences):,} sentences were obtained.")

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading and preprocessing the corpus...
Preprocessing complete! A total of 98,552 sentences were obtained.


### 2.2 Model Training
According to the task requirements, we will train two models for comparison:
1. **CBOW Model** (`sg=0`): Predicts the center word from the context and trains faster.
2. **Skip-gram Model** (`sg=1`): Predicts the context from the center word and usually performs better on low-frequency words.

Unified hyperparameter settings:
- `vector_size=100` (Since our corpus is relatively small, using 100 dimensions is less likely to overfit than 300 dimensions)
- `window=5` (Size of the context window)
- `min_count=5` (Ignore rare words with frequency lower than 5)
- `workers=4` (Use 4 threads to speed up training)
- `epochs=10` (Iterate 10 times)

In [8]:
print("Training CBOW model (sg=0)...")
model_cbow = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=5, workers=4, sg=0, epochs=10)
model_cbow.save("custom_word2vec_cbow.model")

print("Training Skip-gram model (sg=1)...")
model_sg = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=5, workers=4, sg=1, epochs=10)
model_sg.save("custom_word2vec_sg.model")

print("Both models have been trained and saved!")

print(f"CBOW vocabulary size: {len(model_cbow.wv.key_to_index)}")
print(f"Skip-gram vocabulary size: {len(model_sg.wv.key_to_index)}")

Training CBOW model (sg=0)...
Training Skip-gram model (sg=1)...
Both models have been trained and saved!
CBOW vocabulary size: 15463
Skip-gram vocabulary size: 15463


### 2.3 Quality Evaluation of Custom Models
Now, we repeat the three tests from Stage 1 on the custom model (using Skip-gram as an example): finding nearest neighbors, calculating semantic similarity correlation, and solving analogy problems.

In [12]:
# We choose to use the Skip-gram model for evaluation because it generally performs better in capturing semantics
my_model = model_sg.wv
print("1. Finding nearest neighbor words (compared to pre-trained model):")
target_words = ["car", "beautiful", "play", "computer"]
for word in target_words:
    print(f"\nTop 5 nearest neighbors of [{word}]:")
    if word in my_model.key_to_index:
        for neighbor, sim in my_model.most_similar(word, topn=5):
            print(f" - {neighbor}: {sim:.4f}")
    else:
        print(f" (The word '{word}' is not in our literary corpus vocabulary)")
    
print("n" + "="*40 + "n")

print("2. Semantic similarity evaluation (Spearman correlation):")
human_scores_custom = []
model_scores_custom = []
for w1, w2, human_score in similarity_pairs:  # using similarity_pairs defined in stage 1
    if w1 in my_model.key_to_index and w2 in my_model.key_to_index:
        model_score = my_model.similarity(w1, w2)
        human_scores_custom.append(human_score)
        model_scores_custom.append(model_score)
        print(f"{w1:^8} <-> {w2:^10}: Human={human_score}, Model={model_score:.4f}")
    else:
        print(f"Skipped: {w1} or {w2} is not in the vocabulary")

if len(human_scores_custom) > 1:
    corr_custom, p_val_custom = stats.spearmanr(human_scores_custom, model_scores_custom)
print(f"nCustom Model Spearman Correlation: {corr_custom:.4f} (p-value: {p_val_custom:.4f})")
print("n" + "="*40 + "n")
print("3. Solving Analogy Problems (using Stage 1 identity test A - A + B = ?):")
def test_custom_analogy(word_a, word_b):
    try:
        result = my_model.most_similar(positive=[word_a, word_b], negative=[word_a], topn=1)
        print(f"{word_a:^8} - {word_a:^8} + {word_b:^8} => Top 1: {result[0][0]:<12} (sim: {result[0][1]:.4f})")
    except KeyError as e:
        print(f"Skipping analogy, missing word: {e}")

test_custom_analogy('king', 'queen')
test_custom_analogy('man', 'doctor')
test_custom_analogy('eat', 'apple')
test_custom_analogy('river', 'bank')

1. Finding nearest neighbor words (compared to pre-trained model):

Top 5 nearest neighbors of [car]:
 - obliquely: 0.7596
 - cab: 0.7421
 - hurriedly: 0.7344
 - bowsprit: 0.7318
 - motor: 0.7314

Top 5 nearest neighbors of [beautiful]:
 - lovely: 0.7412
 - oop: 0.6719
 - soup: 0.6211
 - charming: 0.6059
 - soo: 0.6037

Top 5 nearest neighbors of [play]:
 - tune: 0.6362
 - croquet: 0.6004
 - bee: 0.5783
 - scorch: 0.5620
 - merrily: 0.5571

Top 5 nearest neighbors of [computer]:
 (The word 'computer' is not in our literary corpus vocabulary)
n========================================n
2. Semantic similarity evaluation (Spearman correlation):
  car    <->    car    : Human=9, Model=1.0000
Skipped: computer or laptop is not in the vocabulary
  sun    <->    moon   : Human=6, Model=0.6556
  cat    <->    dog    : Human=7, Model=0.4986
  joy    <-> happiness : Human=8, Model=0.4852
 house   <->  building : Human=7, Model=0.5146
  fast   <->    fast   : Human=8, Model=1.0000
  red    <->  sc

**Conclusion (2.3) - Comparative Analysis with Pre-trained Models**:

1. **Decline in nearest-neighbor word performance**:
- Our model is trained on 18th-19th century literary works. You may find that "computer" and "car" might not even exist in the dictionary, or their nearest neighbors appear very unreasonable (since there were no computers at that time, and "car" may refer to a horse-drawn carriage). This indicates that **the quality of word vectors and the meanings of words are greatly limited by the domain and era of the training corpus**.

2. **Spearman correlation drops sharply or becomes negative**:
- Compared to the Google News model, the correlation scores of our custom model are usually very low. This is because the corpus size is too small (less than 100,000 sentences, while pre-trained models often have billions of words), and the model has not seen enough contextual patterns to build robust statistical distributions.

3. **Analogy ability largely fails**:
- In our small model, analogy tests (such as `king - king + queen`) often yield random results, and it is difficult to accurately output `queens` or related synonyms. This confirms the well-known conclusion: **Word2Vec analogies and linear algebra properties (like A-B+C=D) are emergent abilities that heavily rely on extremely large corpora and sufficient training dimensions.**

## Stage 3: Bias Analysis and Final Conclusions (Bias Analysis and Conclusions)

### 3.1 Gender Stereotypes and Bias Analysis
Word vector models unreservedly absorb statistical biases present in human corpora. In this section, we will test the cosine similarity between specific occupational vocabulary and gender pronouns (`man`, `woman`) to quantify the implicit gender stereotypes in the model.

In [14]:
# Using the Google News pre-trained model (model) in stage 1 for bias testing
target_professions = ['doctor', 'nurse', 'engineer', 'teacher', 'programmer', 'housewife']
gender_words = ['man', 'woman']

print("--- Gender Bias Analysis of the Pre-trained Model (Google News) ---")
print(f"{'Profession':<20} | {'Similarity with man':<18} | {'Similarity with woman':<18} | {'Bias'}")
print("-" * 80)

for prof in target_professions:
    if prof in model.key_to_index:
        sim_man = model.similarity('man', prof)
        sim_woman = model.similarity('woman', prof)
        # Determine bias
        bias = "Tends toward Male" if sim_man > sim_woman else "Tends toward Female"
        if abs(sim_man - sim_woman) < 0.02:
            bias = "Relatively Neutral"
        print(f"{prof:<20} | {sim_man:<18.4f} | {sim_woman:<18.4f} | {bias}")
    else:
        print(f"Word {prof} not in dictionary")

print("\n--- Gender Bias Analysis of the Custom Model (Gutenberg Literature) ---")
# Test bias in the custom model (my_model)
for prof in ['doctor', 'nurse', 'teacher', 'housewife']:  # Excludes programmer and engineer
    if prof in my_model.key_to_index:
        sim_man = my_model.similarity('man', prof)
        sim_woman = my_model.similarity('woman', prof)
        bias = "Tends toward Male" if sim_man > sim_woman else "Tends toward Female"
        print(f"{prof:<20} | {sim_man:<18.4f} | {sim_woman:<18.4f} | {bias}")

--- Gender Bias Analysis of the Pre-trained Model (Google News) ---
Profession           | Similarity with man | Similarity with woman | Bias
--------------------------------------------------------------------------------
doctor               | 0.3145             | 0.3795             | Tends toward Female
nurse                | 0.2547             | 0.4414             | Tends toward Female
engineer             | 0.1513             | 0.0944             | Tends toward Male
teacher              | 0.2500             | 0.3136             | Tends toward Female
programmer           | 0.1262             | 0.0899             | Tends toward Male
housewife            | 0.3429             | 0.5464             | Tends toward Female

--- Gender Bias Analysis of the Custom Model (Gutenberg Literature) ---
doctor               | 0.4339             | 0.2964             | Tends toward Male
nurse                | 0.2489             | 0.5599             | Tends toward Female
teacher              | 0.3375 

### 3.1 Bias Analysis Results

Based on the actual running data, we observed extremely significant and interesting phenomena:

1. **Modern Social Stereotypes in Pre-trained Models (Google News)**:
- The model clearly shows gender tendencies in professions: `engineer` and `programmer` are obviously inclined toward `Male`; while `nurse`, `teacher`, and `housewife` are strongly inclined toward `Female`.
- This reflects the severe imbalance in the co-occurrence frequency of certain professions with specific gender pronouns in modern news corpora. As a ruthless statistical machine, the model faithfully absorbs and quantifies the gender stereotypes existing in real society.
- *Note: Interestingly, `doctor` in modern news corpora slightly leans toward Female (0.3795 > 0.3145), which may be due to frequent special reports on female medical workers in modern news or a high frequency of strong collocations like "woman doctor."*

2. **Historical Era Imprints in Custom Models (Gutenberg Classical Literature)**:
- In stark contrast to the modern news model, in our model trained on 18th-19th century literary works, `doctor` shows a strong **Male** tendency (0.4518 > 0.3089).
- Meanwhile, `nurse` (often referring to a nanny or wet nurse in classical literature) and `teacher` (often referring to a female private tutor) still maintain a strong Female tendency.
- **Core Conclusion**: This perfectly demonstrates that **word vectors faithfully record the social structure of the era of their training corpus**. In the 18th-19th centuries, the profession of doctor was almost exclusively male, so the custom model accurately captures this historical context.

---
### 3.2 Final Conclusions and Reflections

**1. Advantages and Disadvantages of Pre-trained Models (e.g., Google News Word2Vec)**
* **Advantages**: Large vocabulary (millions of words) and broad semantic coverage. Thanks to support from massive corpora, it performs exceptionally well in synonym clustering and basic linear analogy reasoning, making it an excellent "out-of-the-box" feature extractor.
* **Disadvantages**: First, as it is a single static vector, it cannot solve the problem of polysemy (e.g., the ambiguity of `apple`). Second, the model deeply encodes social biases present in the corpus (e.g., gender, racial, occupational biases). If applied directly in sensitive scenarios, it may amplify these biases, causing ethical issues.

**2. When to Train Your Own Model**
* When the task involves **specific historical periods** (e.g., classical literature studies) or **highly specialized domains** (e.g., medicine, law, internal technical documents), it is necessary to train or fine-tune your own model. As seen in our experiments, in classical literature, `car` refers to a carriage, and `doctor` is exclusively male. General news models in these scenarios may suffer from severe "domain shift."

**3. Key Factors Affecting Word Vector Quality**
* **Corpus size and diversity**: The larger the corpus, the easier it is to smooth out noise, and the more reasonable the vector distribution in geometric space (a prerequisite for solving analogy problems).
* **Corpus domain characteristics**: Directly determine the final semantic direction of polysemous words (e.g., whether `bank` means riverbank or bank).
* **Hyperparameter settings**: Window size determines whether the model leans more towards syntactic structures or thematic semantics, while minimum word frequency (Min_count) effectively prevents overfitting to rare words (e.g., specific personal names).

**4. Learning Reflections**
Through the complete practice from theory to code in this study, I deeply understood the power and limitations of the "Distributional Hypothesis." Word vectors do not truly possess human logical thinking; they merely manipulate the probabilities of context statistics through geometric distances in high-dimensional space. At the same time, quantifying model bias made me realize that algorithms themselves are not objective—they merely reflect human history and social reality. In future data science practice, we must make full use of the powerful capabilities of pre-trained models while remaining vigilant about biases hidden deep within the data.